# LeetCode 566 — Reshape the Matrix bằng Python

**Link đề:** https://leetcode.com/problems/reshape-the-matrix/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix, Simulation

```python
class Solution:
    def matrixReshape(self, mat: List[List[int]], r: int, c: int) -> List[List[int]]:
        ...
```

## Mục tiêu học
1. Hiểu ma trận 2 chiều trong Python là **list của list**, và thứ tự "duyệt theo hàng" (row-major).
2. Nắm điều kiện reshape hợp lệ: **số phần tử không đổi** `m·n == r·c`.
3. Thành thạo **ánh xạ chỉ số** giữa 2 chiều và 1 chiều: `k = i·n + j`, ngược lại `i = k // c`, `j = k % c` — dùng `//`, `%`, `divmod`.
4. Viết 4 lời giải: làm phẳng rồi cắt → điền bằng con trỏ hàng/cột → công thức chỉ số → iterator + slicing; so sánh.
5. Trace, chứng minh, test biên; tránh bẫy `[[0]*c]*r` (các hàng dùng chung).

## 1. Đề bài

Trong MATLAB có hàm `reshape` đổi ma trận `m × n` thành ma trận `r × c` mới, **giữ nguyên dữ liệu**.

Cho ma trận `mat` (`m` hàng, `n` cột) và hai số `r`, `c`. Hãy tạo ma trận `r × c` chứa **mọi phần tử của `mat` theo đúng thứ tự duyệt từng hàng** (trái → phải, trên → dưới).

- Nếu reshape **hợp lệ** → trả về ma trận mới.
- Nếu **không hợp lệ** → trả về **ma trận gốc**.

| `mat` | `r` | `c` | Output |
|---|---:|---:|---|
| `[[1,2],[3,4]]` | 1 | 4 | `[[1,2,3,4]]` |
| `[[1,2],[3,4]]` | 2 | 4 | `[[1,2],[3,4]]` (2·4 = 8 ≠ 4 phần tử → trả gốc) |
| `[[1,2],[3,4]]` | 4 | 1 | `[[1],[2],[3],[4]]` |

**Constraints:** `1 <= m, n <= 100`, `-1000 <= mat[i][j] <= 1000`, `1 <= r, c <= 300`.

## 2. Nền tảng Python cần dùng

### 2.1. Ma trận = list của list

`mat[i]` là **hàng** thứ `i` (một list); `mat[i][j]` là phần tử ở hàng `i`, cột `j`. `len(mat)` = số hàng `m`, `len(mat[0])` = số cột `n`.

In [1]:
mat = [[1, 2, 3],
       [4, 5, 6]]
print("số hàng m =", len(mat), "| số cột n =", len(mat[0]))
print("hàng 1:", mat[1])
print("mat[1][2] =", mat[1][2])

số hàng m = 2 | số cột n = 3
hàng 1: [4, 5, 6]
mat[1][2] = 6


### 2.2. Duyệt theo hàng (row-major order)

Hai vòng `for` lồng nhau: ngoài là hàng, trong là cột. Thứ tự đọc chính là thứ tự đề yêu cầu.

In [2]:
order = []
for i in range(len(mat)):
    for j in range(len(mat[0])):
        order.append(mat[i][j])
print(order)       # [1, 2, 3, 4, 5, 6]

[1, 2, 3, 4, 5, 6]


### 2.3. `//`, `%` và `divmod`

- `a // b`: thương nguyên; `a % b`: số dư.
- `divmod(a, b)` trả về **cả hai** `(a // b, a % b)`.

Ví dụ: phần tử thứ `k = 5` trong ma trận có `c = 2` cột nằm ở hàng `5 // 2 = 2`, cột `5 % 2 = 1`.

In [3]:
print(5 // 2, 5 % 2, divmod(5, 2))
for k in range(6):
    print(f"k={k} -> (hàng, cột) = {divmod(k, 2)} khi c=2")

2 1 (2, 1)
k=0 -> (hàng, cột) = (0, 0) khi c=2
k=1 -> (hàng, cột) = (0, 1) khi c=2
k=2 -> (hàng, cột) = (1, 0) khi c=2
k=3 -> (hàng, cột) = (1, 1) khi c=2
k=4 -> (hàng, cột) = (2, 0) khi c=2
k=5 -> (hàng, cột) = (2, 1) khi c=2


### 2.4. Bẫy kinh điển: `[[0] * c] * r`

`[[0] * c] * r` tạo `r` **tham chiếu tới cùng một hàng**: sửa một hàng thì mọi hàng đổi theo. Cách đúng: list comprehension `[[0] * c for _ in range(r)]` tạo `r` hàng riêng biệt.

In [4]:
bad = [[0] * 3] * 2
bad[0][0] = 9
print("sai :", bad)                    # cả hai hàng đều có 9!

good = [[0] * 3 for _ in range(2)]
good[0][0] = 9
print("đúng:", good)
print("bad[0] is bad[1]?", bad[0] is bad[1], "| good[0] is good[1]?", good[0] is good[1])

sai : [[9, 0, 0], [9, 0, 0]]
đúng: [[9, 0, 0], [0, 0, 0]]
bad[0] is bad[1]? True | good[0] is good[1]? False


`a is b` kiểm tra hai biến có trỏ tới **cùng một object** không (khác `==` so sánh giá trị).

## 3. Khi nào reshape hợp lệ?

Reshape chỉ **sắp xếp lại** các phần tử, không thêm/bớt. Ma trận gốc có `m·n` phần tử, ma trận mới có `r·c` ô ⇒ hợp lệ **khi và chỉ khi** `m · n == r · c`.

| `m×n` | `r×c` | Hợp lệ? |
|---|---|:---:|
| 2×2 = 4 | 1×4 = 4 | ✔ |
| 2×2 = 4 | 2×4 = 8 | ✘ |
| 2×3 = 6 | 3×2 = 6 | ✔ |
| 2×3 = 6 | 6×1 = 6 | ✔ |

In [5]:
def can_reshape(mat, r, c) -> bool:
    return len(mat) * len(mat[0]) == r * c

M = [[1, 2], [3, 4]]
print(can_reshape(M, 1, 4), can_reshape(M, 2, 4), can_reshape(M, 4, 1))

True False True


## 4. Cách 1 — Làm phẳng rồi cắt thành hàng (trực giác)

Cách nghĩ tự nhiên nhất:
1. **Làm phẳng** (flatten) ma trận thành list 1 chiều theo thứ tự hàng.
2. Cắt list đó thành `r` đoạn, mỗi đoạn `c` phần tử: hàng `i` = `flat[i*c : (i+1)*c]`.

Làm phẳng gọn bằng **list comprehension hai vòng**: `[x for row in mat for x in row]` — đọc từ trái sang: "với mỗi `row` trong `mat`, với mỗi `x` trong `row`, lấy `x`".

In [6]:
mat = [[1, 2, 3], [4, 5, 6]]
flat = [x for row in mat for x in row]
print("flat:", flat)
r, c = 3, 2
print([flat[i * c:(i + 1) * c] for i in range(r)])

flat: [1, 2, 3, 4, 5, 6]
[[1, 2], [3, 4], [5, 6]]


In [7]:
def matrix_reshape_flatten(mat: list[list[int]], r: int, c: int) -> list[list[int]]:
    m, n = len(mat), len(mat[0])
    if m * n != r * c:
        return mat
    flat = [x for row in mat for x in row]
    return [flat[i * c:(i + 1) * c] for i in range(r)]

print(matrix_reshape_flatten([[1, 2], [3, 4]], 1, 4))
print(matrix_reshape_flatten([[1, 2], [3, 4]], 2, 4))
print(matrix_reshape_flatten([[1, 2], [3, 4]], 4, 1))

[[1, 2, 3, 4]]
[[1, 2], [3, 4]]
[[1], [2], [3], [4]]


### Code nộp LeetCode — Cách 1

In [8]:
from typing import List

class Solution:
    def matrixReshape(self, mat: List[List[int]], r: int, c: int) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        if m * n != r * c:
            return mat
        flat = [x for row in mat for x in row]
        return [flat[i * c:(i + 1) * c] for i in range(r)]

print(Solution().matrixReshape([[1, 2], [3, 4]], 1, 4))

[[1, 2, 3, 4]]


Độ phức tạp: `O(m·n)` thời gian; bộ nhớ thêm `O(m·n)` cho `flat` (ngoài ma trận kết quả). Cách này dễ hiểu nhưng tạo một list trung gian — ta có thể bỏ nó.

## 5. Cách 2 — Điền trực tiếp bằng con trỏ hàng/cột (mô phỏng)

Ý tưởng: duyệt `mat` theo hàng, đồng thời giữ **vị trí ghi** `(row, col)` trong ma trận mới. Mỗi lần ghi xong, tăng `col`; nếu `col == c` thì xuống hàng: `col = 0`, `row += 1`.

Giống như viết chữ trên giấy có `c` ô mỗi dòng: hết dòng thì xuống dòng.

In [9]:
def matrix_reshape_pointer(mat: list[list[int]], r: int, c: int) -> list[list[int]]:
    m, n = len(mat), len(mat[0])
    if m * n != r * c:
        return mat
    res = [[0] * c for _ in range(r)]
    row = col = 0
    for i in range(m):
        for j in range(n):
            res[row][col] = mat[i][j]
            col += 1
            if col == c:
                col = 0
                row += 1
    return res

print(matrix_reshape_pointer([[1, 2, 3], [4, 5, 6]], 3, 2))

[[1, 2], [3, 4], [5, 6]]


## 6. Trace Cách 2

`mat = [[1,2,3],[4,5,6]]` (2×3) → `r=3, c=2`.

| (i, j) | giá trị | ghi vào (row, col) | col sau | xuống hàng? |
|---|---:|---|---:|:---:|
| (0,0) | 1 | (0,0) | 1 | |
| (0,1) | 2 | (0,1) | 2 → 0 | ✔ row=1 |
| (0,2) | 3 | (1,0) | 1 | |
| (1,0) | 4 | (1,1) | 2 → 0 | ✔ row=2 |
| (1,1) | 5 | (2,0) | 1 | |
| (1,2) | 6 | (2,1) | 2 → 0 | ✔ row=3 |

Kết quả `[[1,2],[3,4],[5,6]]`.

In [10]:
def trace_reshape(mat, r, c):
    m, n = len(mat), len(mat[0])
    print(f"{m}x{n} -> {r}x{c}: {'hợp lệ' if m * n == r * c else 'KHÔNG hợp lệ -> trả gốc'}")
    if m * n != r * c:
        return mat
    res = [[0] * c for _ in range(r)]
    row = col = 0
    for i in range(m):
        for j in range(n):
            res[row][col] = mat[i][j]
            note = f"mat[{i}][{j}]={mat[i][j]} -> res[{row}][{col}]"
            col += 1
            if col == c:
                col, row = 0, row + 1
                note += "  (xuống hàng)"
            print(note, "|", res)
    return res

trace_reshape([[1, 2, 3], [4, 5, 6]], 3, 2)
print()
trace_reshape([[1, 2], [3, 4]], 2, 4)

2x3 -> 3x2: hợp lệ
mat[0][0]=1 -> res[0][0] | [[1, 0], [0, 0], [0, 0]]
mat[0][1]=2 -> res[0][1]  (xuống hàng) | [[1, 2], [0, 0], [0, 0]]
mat[0][2]=3 -> res[1][0] | [[1, 2], [3, 0], [0, 0]]
mat[1][0]=4 -> res[1][1]  (xuống hàng) | [[1, 2], [3, 4], [0, 0]]
mat[1][1]=5 -> res[2][0] | [[1, 2], [3, 4], [5, 0]]
mat[1][2]=6 -> res[2][1]  (xuống hàng) | [[1, 2], [3, 4], [5, 6]]

2x2 -> 2x4: KHÔNG hợp lệ -> trả gốc


[[1, 2], [3, 4]]

## 7. Insight — Công thức ánh xạ chỉ số (gợi ý của LeetCode)

Đánh số mỗi phần tử theo thứ tự hàng bằng **một số duy nhất** `k` (từ 0):

- Từ ma trận gốc `m × n`: phần tử `(i, j)` có `k = i·n + j` (trước nó có `i` hàng đầy, mỗi hàng `n` phần tử, cộng thêm `j`).
- Sang ma trận mới `r × c`: ô thứ `k` nằm ở hàng `k // c`, cột `k % c`.

Ngược lại, ô `(i, j)` của ma trận mới lấy giá trị từ `k = i·c + j` của gốc, tức `mat[k // n][k % n]`.

In [11]:
mat = [[1, 2, 3], [4, 5, 6]]
m, n = 2, 3
r, c = 3, 2
for k in range(m * n):
    src = divmod(k, n)
    dst = divmod(k, c)
    print(f"k={k}: gốc (i,j)={src} giá trị {mat[src[0]][src[1]]} -> mới (i,j)={dst}")

k=0: gốc (i,j)=(0, 0) giá trị 1 -> mới (i,j)=(0, 0)
k=1: gốc (i,j)=(0, 1) giá trị 2 -> mới (i,j)=(0, 1)
k=2: gốc (i,j)=(0, 2) giá trị 3 -> mới (i,j)=(1, 0)
k=3: gốc (i,j)=(1, 0) giá trị 4 -> mới (i,j)=(1, 1)
k=4: gốc (i,j)=(1, 1) giá trị 5 -> mới (i,j)=(2, 0)
k=5: gốc (i,j)=(1, 2) giá trị 6 -> mới (i,j)=(2, 1)


## 8. Cách 3 — Một vòng lặp theo `k` (lời giải chính)

Chỉ một vòng `for k in range(m*n)`:

```python
res[k // c][k % c] = mat[k // n][k % n]
```

Không cần biến `row/col` hay list trung gian.

In [12]:
def matrix_reshape_index(mat: list[list[int]], r: int, c: int) -> list[list[int]]:
    m, n = len(mat), len(mat[0])
    if m * n != r * c:
        return mat
    res = [[0] * c for _ in range(r)]
    for k in range(m * n):
        res[k // c][k % c] = mat[k // n][k % n]
    return res

print(matrix_reshape_index([[1, 2], [3, 4]], 1, 4))
print(matrix_reshape_index([[1, 2, 3], [4, 5, 6]], 3, 2))
print(matrix_reshape_index([[1, 2, 3], [4, 5, 6]], 1, 6))

[[1, 2, 3, 4]]
[[1, 2], [3, 4], [5, 6]]
[[1, 2, 3, 4, 5, 6]]


### Code nộp LeetCode — Công thức chỉ số

In [13]:
class Solution:
    def matrixReshape(self, mat: List[List[int]], r: int, c: int) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        if m * n != r * c:
            return mat
        res = [[0] * c for _ in range(r)]
        for k in range(m * n):
            res[k // c][k % c] = mat[k // n][k % n]
        return res

sol = Solution()
print(sol.matrixReshape([[1, 2], [3, 4]], 1, 4))
print(sol.matrixReshape([[1, 2], [3, 4]], 2, 4))

[[1, 2, 3, 4]]
[[1, 2], [3, 4]]


## 9. Cách 4 — Iterator + `itertools` (gọn kiểu Python)

- `itertools.chain.from_iterable(mat)` nối các hàng thành **một dòng chảy** phần tử (không tạo list).
- `iter(...)` tạo iterator; `next(it)` lấy phần tử kế tiếp.
- `itertools.islice(it, c)` lấy **c phần tử tiếp theo** từ iterator.

Ghép: tạo iterator phẳng, rồi lấy `c` phần tử cho mỗi hàng, lặp `r` lần.

In [14]:
from itertools import chain, islice

it = iter(chain.from_iterable([[1, 2, 3], [4, 5, 6]]))
print(next(it), next(it))          # 1 2
print(list(islice(it, 2)))         # [3, 4]  (tiếp tục từ chỗ cũ)
print(list(it))                    # [5, 6]

1 2
[3, 4]
[5, 6]


In [15]:
def matrix_reshape_iter(mat: list[list[int]], r: int, c: int) -> list[list[int]]:
    if len(mat) * len(mat[0]) != r * c:
        return mat
    it = chain.from_iterable(mat)
    return [list(islice(it, c)) for _ in range(r)]

print(matrix_reshape_iter([[1, 2, 3], [4, 5, 6]], 3, 2))

[[1, 2], [3, 4], [5, 6]]


Iterator "nhớ vị trí": mỗi lần `islice(it, c)` tiếp tục từ chỗ lần trước dừng, nên các hàng nối tiếp nhau đúng thứ tự.

## 10. Vì sao đúng?

Đặt `f(i, j) = i·n + j` cho ma trận gốc. Khi `j` chạy `0..n−1` và `i` chạy `0..m−1`, `f` nhận **mỗi giá trị** `0..m·n−1` đúng một lần, và **tăng dần theo thứ tự duyệt hàng** ⇒ `k` chính là "số thứ tự" của phần tử trong thứ tự row-major.

Tương tự, `g(i, j) = i·c + j` là số thứ tự của ô trong ma trận mới. Phép `(k // c, k % c)` là nghịch đảo của `g` (vì `0 ≤ k % c < c`).

Cách 3 đặt phần tử số thứ tự `k` của gốc vào ô số thứ tự `k` của ma trận mới ⇒ thứ tự duyệt hàng được giữ nguyên, đúng yêu cầu. Điều kiện `m·n == r·c` đảm bảo mọi ô được điền đúng một lần, không thừa không thiếu.

## 11. So sánh và độ phức tạp

Gọi `N = m·n` (số phần tử).

| Cách | Time | Space thêm (ngoài kết quả) | Ghi chú |
|---|---|---|---|
| 1. Làm phẳng + cắt | `O(N)` | `O(N)` (list `flat`) | dễ hiểu nhất |
| 2. Con trỏ hàng/cột | `O(N)` | `O(1)` | mô phỏng "xuống dòng" |
| 3. Công thức chỉ số | `O(N)` | `O(1)` | **gọn, kinh điển** |
| 4. Iterator `chain`/`islice` | `O(N)` | `O(1)` | phong cách Python |

Trường hợp không hợp lệ: `O(1)` (chỉ so sánh tích).

## 12. Bộ test: mẫu đề, biên, ngẫu nhiên

In [16]:
tests = [
    ([[1, 2], [3, 4]], 1, 4, [[1, 2, 3, 4]]),
    ([[1, 2], [3, 4]], 2, 4, [[1, 2], [3, 4]]),
    ([[1, 2], [3, 4]], 4, 1, [[1], [2], [3], [4]]),
    ([[1, 2], [3, 4]], 2, 2, [[1, 2], [3, 4]]),          # giữ nguyên kích thước
    ([[1, 2, 3], [4, 5, 6]], 3, 2, [[1, 2], [3, 4], [5, 6]]),
    ([[1, 2, 3, 4]], 2, 2, [[1, 2], [3, 4]]),
    ([[7]], 1, 1, [[7]]),
    ([[7]], 1, 2, [[7]]),                                  # không hợp lệ
    ([[1, 2, 3], [4, 5, 6]], 4, 2, [[1, 2, 3], [4, 5, 6]]),
]
methods = [matrix_reshape_flatten, matrix_reshape_pointer, matrix_reshape_index, matrix_reshape_iter]
for mat, r, c, expected in tests:
    for f in methods:
        assert f(mat, r, c) == expected, (f.__name__, mat, r, c)
    print(f"{str(mat):<26} r={r} c={c} -> {expected}")
print("Test mẫu PASS!")

[[1, 2], [3, 4]]           r=1 c=4 -> [[1, 2, 3, 4]]
[[1, 2], [3, 4]]           r=2 c=4 -> [[1, 2], [3, 4]]
[[1, 2], [3, 4]]           r=4 c=1 -> [[1], [2], [3], [4]]
[[1, 2], [3, 4]]           r=2 c=2 -> [[1, 2], [3, 4]]
[[1, 2, 3], [4, 5, 6]]     r=3 c=2 -> [[1, 2], [3, 4], [5, 6]]
[[1, 2, 3, 4]]             r=2 c=2 -> [[1, 2], [3, 4]]
[[7]]                      r=1 c=1 -> [[7]]
[[7]]                      r=1 c=2 -> [[7]]
[[1, 2, 3], [4, 5, 6]]     r=4 c=2 -> [[1, 2, 3], [4, 5, 6]]
Test mẫu PASS!


In [17]:
import random

random.seed(566)
for _ in range(400):
    m, n = random.randint(1, 6), random.randint(1, 6)
    mat = [[random.randint(-1000, 1000) for _ in range(n)] for _ in range(m)]
    r, c = random.randint(1, 12), random.randint(1, 12)
    answers = [f(mat, r, c) for f in methods]
    assert all(a == answers[0] for a in answers), (mat, r, c)
    if m * n == r * c:
        # tính chất: reshape ngược lại về m x n phải ra đúng ma trận gốc
        assert matrix_reshape_index(answers[0], m, n) == mat
print("400 test ngẫu nhiên PASS (4 cách khớp nhau, reshape ngược khôi phục gốc).")

400 test ngẫu nhiên PASS (4 cách khớp nhau, reshape ngược khôi phục gốc).


`all(...)` trả `True` nếu **mọi** phần tử đúng. Tính chất "reshape đi rồi reshape về" là một phép kiểm tra rất mạnh: nếu thứ tự bị sai ở đâu, ma trận khôi phục sẽ khác gốc.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Tạo ma trận bằng `[[0] * c] * r`

In [18]:
def wrong_shared_rows(mat, r, c):
    m, n = len(mat), len(mat[0])
    if m * n != r * c:
        return mat
    res = [[0] * c] * r                   # SAI: r hàng là cùng một list
    for k in range(m * n):
        res[k // c][k % c] = mat[k // n][k % n]
    return res

print("sai :", wrong_shared_rows([[1, 2, 3], [4, 5, 6]], 3, 2))
print("đúng:", matrix_reshape_index([[1, 2, 3], [4, 5, 6]], 3, 2))

sai : [[5, 6], [5, 6], [5, 6]]
đúng: [[1, 2], [3, 4], [5, 6]]


Vì sao sai: mọi hàng là một object → lần ghi cuối đè lên tất cả. Sửa: `[[0] * c for _ in range(r)]`.

### Lỗi 2: Chia/lấy dư nhầm số cột

Đọc gốc phải dùng số cột **gốc** `n`: `mat[k // n][k % n]`; ghi mới dùng số cột **mới** `c`: `res[k // c][k % c]`. Nhầm `n` ↔ `c` → sai hoặc `IndexError`.

In [19]:
def wrong_swap_nc(mat, r, c):
    m, n = len(mat), len(mat[0])
    if m * n != r * c:
        return mat
    res = [[0] * c for _ in range(r)]
    for k in range(m * n):
        res[k // c][k % c] = mat[k // c][k % c]     # SAI: phải là n khi đọc mat
    return res

try:
    print(wrong_swap_nc([[1, 2, 3], [4, 5, 6]], 3, 2))
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### Lỗi 3: Kiểm tra hợp lệ bằng `m == r` hoặc `n == c`

Điều kiện đúng là **tích** bằng nhau, không phải từng chiều. `2×3 → 3×2` hợp lệ dù `m ≠ r`.

### Lỗi 4: Trả về `None` hoặc `[]` khi không hợp lệ

Đề yêu cầu trả **ma trận gốc**. Đọc kỹ đề!

### Lỗi 5: Dùng `/` thay vì `//`

`5 / 2 = 2.5` (float) không dùng làm chỉ số được → `TypeError`.

In [20]:
row = [10, 20, 30]
try:
    print(row[5 / 2])
except TypeError as e:
    print("TypeError:", e)
print(row[5 // 2])

TypeError: list indices must be integers or slices, not float
30


## 14. Mẹo nhận dạng pattern và biến thể

### Mẹo
- Bài đổi hình dạng, xoay, duyệt ma trận theo "số thứ tự" → **ánh xạ 1D ↔ 2D**: `k = i·cols + j`, `(i, j) = divmod(k, cols)`.
- Cần duyệt nối tiếp nhiều list như một dòng → `itertools.chain`.
- Tạo ma trận rỗng: **luôn** `[[0] * c for _ in range(r)]`.

### Biến thể
1. **LeetCode 867 – Transpose Matrix**: `res[j][i] = mat[i][j]`.
2. **LeetCode 74 – Search a 2D Matrix** (Medium): tìm kiếm nhị phân trên ma trận coi như mảng 1D dùng `divmod`.
3. **LeetCode 2022 – Convert 1D Array Into 2D Array**: ngược của làm phẳng.
4. **LeetCode 1260 – Shift 2D Grid**: dịch phần tử theo số thứ tự `k`.

In [21]:
# Biến thể 2 demo: binary search trên ma trận đã sắp theo hàng, coi như mảng 1D
def search_matrix(mat, target):
    m, n = len(mat), len(mat[0])
    lo, hi = 0, m * n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        val = mat[mid // n][mid % n]
        if val == target:
            return True
        if val < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

grid = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
print(search_matrix(grid, 3), search_matrix(grid, 13))

True False


## 15. Bài tập tự luyện

### Bài 1 (dễ)
Viết `flatten(mat)` trả về list 1 chiều theo thứ tự hàng, **dùng hai vòng `for`** (không dùng comprehension).

In [22]:
def flatten(mat) -> list:
    # TODO
    pass

# assert flatten([[1, 2], [3, 4]]) == [1, 2, 3, 4]

### Bài 2 (dễ)
Viết `construct_2d(original, m, n)` (LeetCode 2022): biến list 1D thành ma trận `m × n`; nếu không đủ/thừa phần tử trả `[]`.

In [23]:
def construct_2d(original: list, m: int, n: int) -> list:
    # TODO
    pass

# assert construct_2d([1, 2, 3, 4], 2, 2) == [[1, 2], [3, 4]]
# assert construct_2d([1, 2], 1, 1) == []

### Bài 3 (vừa)
Viết `transpose(mat)`: ma trận chuyển vị (`n × m`), `res[j][i] = mat[i][j]`.

In [24]:
def transpose(mat) -> list:
    # TODO
    pass

# assert transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]

### Bài 4 (vừa)
Viết `reshape_column_major(mat, r, c)`: giống bài gốc nhưng **đọc và điền theo cột** (trên → dưới, rồi sang cột kế tiếp), như MATLAB thật.

In [25]:
def reshape_column_major(mat, r: int, c: int) -> list:
    # TODO
    pass

# assert reshape_column_major([[1, 2, 3], [4, 5, 6]], 3, 2) == [[1, 5], [4, 3], [2, 6]]

### Bài 5 (khó hơn)
Viết `shift_grid(grid, k)` (LeetCode 1260): dịch mọi phần tử sang phải `k` bước theo thứ tự hàng (phần tử cuối quay về đầu).

In [26]:
def shift_grid(grid, k: int) -> list:
    # TODO
    pass

# assert shift_grid([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1) == [[9, 1, 2], [3, 4, 5], [6, 7, 8]]

## 16. Gợi ý 3 mức

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | List rỗng `out` | Vòng ngoài hàng, trong phần tử | `out.append(x)` |
| 2 | Kiểm tra `len == m*n` | Cắt theo đoạn `n` | `[original[i*n:(i+1)*n] for i in range(m)]` |
| 3 | Kết quả có `n` hàng | Hàng `j` của kết quả = cột `j` của gốc | `[[mat[i][j] for i in range(m)] for j in range(n)]` |
| 4 | Số thứ tự theo cột: `k = j·m + i` | Ngược lại: `i = k % rows`, `j = k // rows` | Áp cho cả gốc (rows=m) và mới (rows=r) |
| 5 | Làm phẳng | Vị trí mới `(k + shift) % N` | Hoặc `flat[-k:] + flat[:-k]` rồi cắt lại |

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [27]:
def answer_flatten(mat) -> list:
    out = []
    for row in mat:
        for x in row:
            out.append(x)
    return out

def answer_construct_2d(original: list, m: int, n: int) -> list:
    if len(original) != m * n:
        return []
    return [original[i * n:(i + 1) * n] for i in range(m)]

def answer_transpose(mat) -> list:
    m, n = len(mat), len(mat[0])
    return [[mat[i][j] for i in range(m)] for j in range(n)]

def answer_reshape_column_major(mat, r: int, c: int) -> list:
    m, n = len(mat), len(mat[0])
    if m * n != r * c:
        return mat
    res = [[0] * c for _ in range(r)]
    for k in range(m * n):
        res[k % r][k // r] = mat[k % m][k // m]
    return res

def answer_shift_grid(grid, k: int) -> list:
    m, n = len(grid), len(grid[0])
    N = m * n
    k %= N
    flat = answer_flatten(grid)
    flat = flat[N - k:] + flat[:N - k]
    return answer_construct_2d(flat, m, n)

assert answer_flatten([[1, 2], [3, 4]]) == [1, 2, 3, 4]
assert answer_construct_2d([1, 2, 3, 4], 2, 2) == [[1, 2], [3, 4]]
assert answer_construct_2d([1, 2], 1, 1) == []
assert answer_transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]]
assert answer_reshape_column_major([[1, 2, 3], [4, 5, 6]], 3, 2) == [[1, 5], [4, 3], [2, 6]]
assert answer_shift_grid([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1) == [[9, 1, 2], [3, 4, 5], [6, 7, 8]]
assert answer_shift_grid([[1, 2], [3, 4]], 4) == [[1, 2], [3, 4]]
# reshape theo cột rồi reshape theo cột ngược về m x n phải khôi phục gốc
for _ in range(100):
    m, n = random.randint(1, 5), random.randint(1, 5)
    mat = [[random.randint(0, 9) for _ in range(n)] for _ in range(m)]
    assert answer_reshape_column_major(answer_reshape_column_major(mat, n, m), m, n) == mat
print("Đáp án chạy PASS toàn bộ test!")

Đáp án chạy PASS toàn bộ test!


## 18. Template nhớ nhanh và checklist

```python
class Solution:
    def matrixReshape(self, mat: List[List[int]], r: int, c: int) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        if m * n != r * c:
            return mat                                  # không hợp lệ -> trả gốc
        res = [[0] * c for _ in range(r)]               # KHÔNG dùng [[0]*c]*r
        for k in range(m * n):
            res[k // c][k % c] = mat[k // n][k % n]     # đọc theo n, ghi theo c
        return res
```

Công thức cần thuộc: `k = i·cols + j` ⇔ `(i, j) = divmod(k, cols)`.

### Checklist trước khi submit
- [ ] Kiểm tra `m * n == r * c`; sai thì trả **`mat` gốc**.
- [ ] Tạo ma trận bằng comprehension, không nhân list lồng.
- [ ] Đọc gốc dùng `n`, ghi mới dùng `c`.
- [ ] Dùng `//` và `%`, không dùng `/`.
- [ ] Đã thử `r×c` giống hệt `m×n`, ma trận 1×1, một hàng, một cột.